In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/store-sales-time-series-forecasting/train.csv')
test = pd.read_csv('/kaggle/input/competitions/store-sales-time-series-forecasting/test.csv')
oil = pd.read_csv('/kaggle/input/competitions/store-sales-time-series-forecasting/oil.csv')
stores = pd.read_csv('/kaggle/input/competitions/store-sales-time-series-forecasting/stores.csv')
transactions = pd.read_csv('/kaggle/input/competitions/store-sales-time-series-forecasting/transactions.csv')
holiday = pd.read_csv('/kaggle/input/competitions/store-sales-time-series-forecasting/holidays_events.csv')

In [ ]:
pip install skforecast

In [ ]:

from lightgbm import LGBMRegressor
from skforecast.recursive import ForecasterRecursive
from sklearn.preprocessing import LabelEncoder

In [ ]:
estimator_params = {
    'n_estimators': 100,
    'learning_rate': 0.1,
    'max_depth': 6,
    'num_leaves': 31,
    'random_state': 123,
    'verbose': -1
}

exog_features = [ "month_sin", "month_cos", "quarter_sin", "quarter_cos", "day_of_week_sin", "day_of_week_cos", "weekend"]


In [ ]:
def pre(data):
    data = data.copy()
    data['month_sin'] = np.sin(2 * np.pi * data.index.month / 12)
    data['month_cos'] = np.cos(2 * np.pi * data.index.month / 12)
    data['quarter_sin'] = np.sin(2 * np.pi * data.index.quarter / 4)
    data['quarter_cos'] = np.cos(2 * np.pi * data.index.quarter / 4)
    data['day_of_week_sin'] = np.sin(2 * np.pi * data.index.dayofweek / 7)
    data['day_of_week_cos'] = np.cos(2 * np.pi * data.index.dayofweek / 7)
    data['weekend'] = data.index.dayofweek.isin([5, 6]).astype(int)
    return data

In [ ]:
results = []

for (store_nbr, family), _ in test.groupby(["store_nbr", "family"]):

    # ── training data for this group ─────────────────────────────────────────
    data = (
        train[
            (train["store_nbr"] == store_nbr) &
            (train["family"]    == family)
        ]
        .copy()
    )


    # ── test data for this group ──────────────────────────────────────────────
    test_group = (
        test[
            (test["store_nbr"] == store_nbr) &
            (test["family"]    == family)
        ]
        .copy()
    )
    test_group["date"] = pd.to_datetime(test_group["date"])
    test_group = test_group.set_index("date").asfreq("D").sort_index()

    data['date'] = pd.to_datetime(data['date'])
    data = data.set_index('date').asfreq('D').sort_index()


    data['sales'] = data['sales'].fillna(0)

    # ── add calendar features (after asfreq so no zero-filled gaps) ───────────
    data       = pre(data)
    test_group = pre(test_group)

    # ── fit & predict ─────────────────────────────────────────────────────────
    forecaster = ForecasterRecursive(
        estimator=LGBMRegressor(**estimator_params),
        lags=24,
    )

    forecaster.fit(
        y=data["sales"],
        exog=data[exog_features],
    )

    preds = forecaster.predict(
        steps=len(test_group),
        exog=test_group[exog_features],
    )

    # ── collect ───────────────────────────────────────────────────────────────
    temp_df = test_group.copy()
    temp_df["store_nbr"] = store_nbr
    temp_df["family"]    = family
    temp_df["pred"]      = preds.values.clip(0)
    results.append(temp_df)
    print(f"✓ store={store_nbr} family={family}  steps={len(preds)}")

final_df = pd.concat(results, ignore_index=False)  # keep DatetimeIndex
print(f"\nDone — {len(final_df)} rows across {len(results)} groups")
print(final_df[["store_nbr", "family", "pred"]].head(20))

In [ ]:
from IPython.display import FileLink
df.to_csv('1_bev.csv', index=False)
#final_df[['id','pred']].rename(columns={'pred':'sales'}).to_csv('submission.csv', index=False)
FileLink('1_bev.csv')

In [ ]:
final_df[['id','pred']].rename(columns={'pred':'sales'}).to_csv('submission.csv', index=False)

In [ ]:
submission = pd.concat(all_preds, ignore_index=True)
submission = submission.sort_values("id")[["id", "sales"]]
submission.to_csv("submission.csv", index=False)
print(f"Done — {len(submission)} rows")